# Inicialização

In [1]:
import importlib
from pathlib import Path
import os
import sys

# Detecta se o notebook está rodando no modo "Save Version" (Background)
# O Kaggle define a variável KAGGLE_URL_CPU quando está em execução em background
IS_BACKGROUND = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') == 'Batch'

if IS_BACKGROUND:
    print("--- Executando em modo SAVE (Background) ---")
    # 1. Força o sistema a enxergar apenas uma GPU, matando o conflito de registro cuBLAS/cuFFT
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["PAPERMILL_IOPUB_TIMEOUT"] = "600"
    
    # 3. Força o PyTorch a não usar multiprocessamento na inicialização de drivers
    import torch
    torch.set_num_threads(1)
    os.environ["PAPERMILL_IOPUB_TIMEOUT"] = "300"

    # Se o nbclient estiver usando variáveis internas, alteramos também via código
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except:
        pass

import logging
import subprocess
import site


# for handler in logging.root.handlers[:]:
#     logging.root.removeHandler(handler)
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

## ---------LOGGING-----------
# Define paths based on environment
if Path("/kaggle").exists():
    print("IN KAGGLE")
    os.environ["AMBIENTE"] = "KAGGLE"
    os.environ["TENSORBOARD_NO_TF"] = "1"

    PATH_DATASET = Path("/kaggle/working/TRAB_SERIES_TEMP")
    PATH_CODE = PATH_DATASET
    PATH_OUTPUT_DIR = PATH_DATASET / "outputs"
    EXPORT_DIR = PATH_OUTPUT_DIR / "export"
elif Path("/content").exists():
    os.environ["AMBIENTE"] = "COLAB"
    PATH_DATASET = Path("/content/TRAB_SERIES_TEMP")
    PATH_CODE = PATH_DATASET / "src"
    PATH_OUTPUT_DIR = PATH_DATASET / "outputs"
    EXPORT_DIR = PATH_OUTPUT_DIR / "export"
else:
    os.environ["AMBIENTE"] = "LOCAL"
    PATH_CODE = Path.cwd()
    PATH_DATASET = PATH_CODE
    PATH_OUTPUT_DIR = PATH_DATASET / "outputs"
    # Add to setup cell
    EXPORT_DIR = PATH_OUTPUT_DIR / "export"

assert type(PATH_DATASET) is not str, "PATH_DATASET NÃO DEVE SER STR!"
# Check if installation has been done
INSTALL_MARKER = PATH_DATASET / ".install_complete"

try:
    if not INSTALL_MARKER.exists():
        print("Iniciando instalação de dependências...")

        if os.environ["AMBIENTE"] == "KAGGLE":
            # Usando subprocess.run com check=True para travar o código se houver erro
            subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)

            os.chdir("/kaggle/working")
            subprocess.run(
                ["git", "clone", "https://github.com/lfaoliveira/TRAB_SERIES_TEMP.git"], check=True
            ) 
        elif os.environ["AMBIENTE"] == "COLAB":
            subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)

            subprocess.run(
                ["git", "clone", "https://github.com/lfaoliveira/TRAB_SERIES_TEMP.git"], check=True
            )

        os.chdir(PATH_DATASET)

        # Roda a instalação. Se o uv falhar, o check=True aciona um erro
        print("Instalando pacotes do pyproject.toml...")
        subprocess.run(
            ["uv", "pip", "install", "--requirements", "pyproject.toml", "--group", "kaggle", "--system"],
            check=True,
        )
        
        
        # Só marca como completo se NENHUM erro aconteceu acima
        importlib.invalidate_caches()
        importlib.reload(site)
        INSTALL_MARKER.touch()
        print("Installation completed")

    else:
        print("Installation already completed, skipping...")

    os.chdir(PATH_CODE)
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Current working directory: {os.getcwd()}")
except Exception as e:
    print("FALHA AO INICIAR NOTEBOOK")
    print(e)

Installation already completed, skipping...
Current working directory: c:\Users\LUIS FELIPE\Desktop\SERIES_TEMP


## Carregamento

In [2]:
from darts import TimeSeries
import pandas as pd
import numpy as np

from src.data.dataset import NasaDataset
from pathlib import Path
import os
import torch
torch.manual_seed(42)

# from src.data.dataset import NasaDataset
# from src.data.datamodule import StrokeDataModule
import warnings

warnings.filterwarnings("ignore")
# Instanciar dataset (carrega e preprocessa automaticamente)
print("Carregando StrokeDataset...")
if os.environ["AMBIENTE"] == "KAGGLE" or os.environ["AMBIENTE"] == "COLAB":
    # O dataset montado via UI aparecerá aqui (ajuste "nome-do-dataset" para o real)
    PATH_DATA = Path("/kaggle/input/datasets/patrickfleith/nasa-anomaly-detection-dataset-smap-msl")
else:
    PATH_DATA = Path("dataset")

PATH_EXPORT = Path("outputs", "export")
dataset = NasaDataset(PATH_DATA, export_path=PATH_EXPORT, normalize=False)

# build_train_dataset() retorna (train_values, train_labels) como tuple
train_values, train_labels = dataset.build_train_dataset()
test_values, test_labels = dataset.build_test_dataset()


# =====================================================
# PRINT RESUMIDO E EXPORT CSV
# =====================================================
def summarise_series_set(values, labels, name: str):
    """Printa estatísticas resumidas de um conjunto treino/teste."""
    n = len(values)
    print(f"\n{'=' * 60}")
    print(f"  {name.upper()}")
    print(f"{'=' * 60}")
    print(f"  N° séries: {n}")

    # Comprimentos
    lengths = [len(s) for s in values]
    print(f"  Comprimento — min: {min(lengths)}, max: {max(lengths)}, média: {np.mean(lengths):.1f}")

    # Valores (todos os pontos de todas as séries)
    all_vals = np.concatenate([s.values(copy=False).flatten() for s in values])
    all_vals_clean = all_vals[~np.isnan(all_vals)]
    print(
        f"  Valores — min: {all_vals_clean.min():.4f}, max: {all_vals_clean.max():.4f}, "
        f"média: {all_vals_clean.mean():.4f}, std: {all_vals_clean.std():.4f}"
    )
    n_nan = np.isnan(all_vals).sum()
    print(f"  NaN: {n_nan} ({n_nan / len(all_vals) * 100:.2f}%)")

    # Labels (anomalias)
    all_lbl = np.concatenate([label.values(copy=False).flatten() for label in labels])
    n_anom = int(all_lbl.sum())
    print(f"  Anomalias: {n_anom} ({all_lbl.mean() * 100:.2f}%)")

    return lengths, all_vals_clean, all_lbl


len_train, vals_train, lbl_train = summarise_series_set(train_values, train_labels, "TREINO")
len_test, vals_test, lbl_test = summarise_series_set(test_values, test_labels, "TESTE")


# =====================================================
# EXPORTAR PARA CSV
# =====================================================
print(f"\n{'=' * 60}")
print("  EXPORTANDO CSVs…")
print(f"{'=' * 60}")

PATH_EXPORT.mkdir(parents=True, exist_ok=True)


# Cada TimeSeries vira uma linha no CSV largo, com ID da série como prefixo
def export_series_to_csv(series_list, filename: str):
    rows = []
    for i, ts in enumerate(series_list):
        sid = f"S{i}"
        vals = ts.values(copy=False).flatten()
        rows.append(pd.Series(vals, name=sid))
    df = pd.concat(rows, axis=1)
    df.to_csv(PATH_EXPORT / filename)
    print(f"  → {filename}: {df.shape[0]} timesteps x {df.shape[1]} séries")


export_series_to_csv(train_values, "train_values.csv")
export_series_to_csv(train_labels, "train_labels.csv")
export_series_to_csv(test_values, "test_values.csv")
export_series_to_csv(test_labels, "test_labels.csv")
print(f"  CSVs salvos em: {PATH_EXPORT.resolve()}")

print("\n  ✓ Dataset carregado com sucesso!")
print(f"  TAMANHO DO DATASET: {dataset.tam_dataset}")

INFO:root:run_mode='prototype' dataset=DatasetConfig(dataset_frequency='Daily', impute=False)
Carregando StrokeDataset...
INFO:root:DATASET PATH: dataset\data\data
INFO:root:Loading train split …
INFO:root:Loaded 82 channels from dataset\data\data\data\data\train
INFO:root:DF TRAIN:
                                value
series_id feature_id time_idx       
A-1       feat_0     0         0.999
                     1         0.999
                     2         0.999
                     3         0.999
                     4         0.999
INFO:root:Loading test split …
INFO:root:Loaded 82 channels from dataset\data\data\data\data\test
INFO:root:Loaded 105 anomaly sequences across 81 channels.
INFO:root:Empty DataFrame
Columns: [value]
Index: []
INFO:root:Empty DataFrame
Columns: [value]
Index: []

  TREINO
  N° séries: 82
  Comprimento — min: 312, max: 4308, média: 2399.3
  Valores — min: -1.4772, max: 4.1627, média: -0.1684, std: 0.8254
  NaN: 0 (0.00%)
  Anomalias: 0 (0.00%)

  TESTE


# Exploratory Data Analysis

In [ ]:
# ============================================================
# EXPLORAÇÃO E VISUALIZAÇÃO
# ============================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import acf, pacf


# Função para calcular estatísticas de cada série
def calculate_series_stats(series: TimeSeries):
    """Calcula estatísticas para uma série temporal Darts."""
    # Converte para numpy array e remove NaN corretamente
    values_nan = series.values().flatten()
    values = values_nan[~np.isnan(values_nan)]

    # Estatísticas básicas
    mean = np.mean(values)
    std = np.std(values)
    q1 = np.percentile(values, 25)
    q2 = np.percentile(values, 50)  # mediana
    q3 = np.percentile(values, 75)

    # Porcentagem de NaN
    pct_nan = (np.isnan(values).sum() / len(values)) * 100

    # Elementos acima de 3 desvios padrão
    above_3std = np.sum(np.abs(values - mean) > 3 * std)

    # ACF e PACF (até 20 lags ou menos se a série for menor)
    max_lags = min(20, len(values) // 2)
    if max_lags > 1:
        acf_values = acf(values, nlags=max_lags, fft=True)
        pacf_values = pacf(values, nlags=max_lags)
    else:
        acf_values = np.array([np.nan])
        pacf_values = np.array([np.nan])

    return {
        "mean": mean,
        "std": std,
        "q1": q1,
        "q2": q2,
        "q3": q3,
        "pct_nan": pct_nan,
        "above_3std": above_3std,
        "acf": acf_values,
        "pacf": pacf_values,
        "length": len(values),
    }


# Itera sobre as séries de TREINO (train_values é list[TimeSeries])
print("Calculando estatísticas para cada série de treino...")
stats_list = []

for idx, series in enumerate(train_values):
    stats = calculate_series_stats(series)
    stats["series_id"] = idx
    stats_list.append(stats)

    if (idx + 1) % max(1, int(len(train_values) / 10)) == 0:
        print(f"Progresso: {((idx + 1) / len(train_values)) * 100:.1f}%")

print(f"Total de séries processadas: {len(stats_list)}")

# Cria DataFrame principal
df_stats = pd.DataFrame([{k: v for k, v in s.items() if k not in ["acf", "pacf"]} for s in stats_list])

# Separa ACF e PACF em colunas
df_acf = pd.DataFrame(
    [s["acf"] for s in stats_list],
    columns=[f"acf_lag{i}" for i in range(len(stats_list[0]["acf"]))],
)
df_pacf = pd.DataFrame(
    [s["pacf"] for s in stats_list],
    columns=[f"pacf_lag{i}" for i in range(len(stats_list[0]["pacf"]))],
)

# Concatena tudo
df_stats = pd.concat([df_stats, df_acf, df_pacf], axis=1)

print(f"DataFrame de estatísticas criado: {df_stats.shape}")
print(df_stats.head())


# ============================================================
# VISUALIZAÇÕES
# ============================================================

# --- Gráfico de Média por série com barras de desvio padrão ---
fig_mean_std = go.Figure()

fig_mean_std.add_trace(
    go.Scatter(
        x=df_stats["series_id"],
        y=df_stats["mean"],
        mode="markers",
        name="Média",
        marker=dict(color="royalblue", size=5, opacity=0.7),
        error_y=dict(
            type="data",
            array=df_stats["std"],
            visible=True,
            color="rgba(100,100,100,0.3)",
            thickness=1,
        ),
        hovertemplate="Série %{x}<br>Média=%{y:.4f}±%{customdata:.4f}<extra></extra>",
        customdata=df_stats["std"],
    )
)

fig_mean_std.update_layout(
    title="Média ± Desvio Padrão de Cada Série",
    xaxis_title="ID da Série",
    yaxis_title="Valor",
    width=1400,
    height=500,
    hovermode="closest",
)
fig_mean_std.show()


# --- Quantidade de anomalias por série (test_labels agora é list[TimeSeries]) ---
# Converte cada TimeSeries de label para numpy array
anomaly_counts = np.array([int(lbl_ts.values(copy=False).sum()) for lbl_ts in test_labels])
anomaly_pct = np.array([float(lbl_ts.values(copy=False).mean() * 100) for lbl_ts in test_labels])
series_ids = np.arange(len(test_labels))

fig_anomaly = make_subplots(
    rows=1,
    cols=1,
    subplot_titles=("Contagem de Pontos Anômalos por Série", "% de Anomalias por Série"),
    shared_xaxes=True,
    vertical_spacing=0.12,
)


fig_anomaly.add_trace(
    go.Bar(
        x=series_ids,
        y=anomaly_pct,
        name="% Anomalias",
        marker_color="darkorange",
        opacity=0.75,
        hovertemplate="Série %{x}<br>% Anomalias=%{y:.2f}%<extra></extra>",
    ),
    row=1,
    col=1,
)

fig_anomaly.update_layout(
    title="Distribuição de Anomalias nas Séries de Teste",
    height=550,
    width=1400,
    hovermode="closest",
    showlegend=False,
)
fig_anomaly.update_xaxes(title_text="ID da Série", row=2, col=1)
fig_anomaly.update_yaxes(title_text="Contagem", row=1, col=1)
fig_anomaly.update_yaxes(title_text="%", row=2, col=1)
fig_anomaly.show()

print("\n=== ANOMALIAS ===")
print(f"Séries com anomalias: {(anomaly_counts > 0).sum()} / {len(anomaly_counts)}")
print(f"Total de pontos anômalos: {anomaly_counts.sum()}")
print(f"Média de anomalias por série: {anomaly_counts.mean():.1f}")
print(f"% média de anomalias por série: {anomaly_pct.mean():.2f}%")

# Estatísticas resumidas
print("\n=== RESUMO DAS ESTATÍSTICAS ===")
print(f"Média global: {df_stats['mean'].mean():.4f}")
print(f"Desvio padrão médio: {df_stats['std'].mean():.4f}")
print(f"Séries com > 0% NaN: {(df_stats['pct_nan'] > 0).sum()}")
print(f"Máximo % NaN: {df_stats['pct_nan'].max():.1f}%")
print(f"Média de elementos acima de 3 std: {df_stats['above_3std'].mean():.2f}")

# Treinamento dos modelos

## Modelos de baseline

In [3]:
import gc
import traceback

from src.models.outlier import OutlierDetector
from src.models.baselines import Hampel, KMeans, IsolationForest, LocalOutlierFactor


WINDOW_SIZE = 15
models: list[OutlierDetector] = [
    Hampel(WINDOW_SIZE),
    KMeans(WINDOW_SIZE),
    IsolationForest(WINDOW_SIZE),
    LocalOutlierFactor(WINDOW_SIZE),
]
print("\n✓ Dataset carregado com sucesso!")
print(f"TAMANHO DO DATASET: {dataset.tam_dataset}")
# =====================================================
# TRAIN
# =====================================================
metricas_models = []
for model in models:
    try:
        dict_met = model.apply(train_values, train_labels, test_values, test_labels)
        metricas_models.append(dict_met)
    except Exception as e:
        traceback.print_exc()
        print(model)
        raise e
    gc.collect()
# Cada dict_met é {nome_modelo: {auc_roc, auc_pr}} — achatar
flat_records = []
for d in metricas_models:
    for nome, met in d.items():
        flat_records.append(met)
res = pd.DataFrame(flat_records)
print(res)


✓ Dataset carregado com sucesso!
TAMANHO DO DATASET: (510225, 1)
INFO:root:MODELO: Hampel
INFO:root:TREINANDO ...
INFO:root:TESTANDO ...
INFO:root:METRIFICANDO ...
INFO:root:FINALIZADO!

INFO:root:MODELO: KMeans
INFO:root:TREINANDO ...
INFO:root:TESTANDO ...
INFO:root:METRIFICANDO ...
INFO:root:FINALIZADO!

INFO:root:MODELO: IsolationForest
INFO:root:TREINANDO ...
INFO:root:TESTANDO ...
INFO:root:METRIFICANDO ...
INFO:root:FINALIZADO!

INFO:root:MODELO: LocalOutlierFactor
INFO:root:TREINANDO ...


: 

: 

# Modelos Rede Neural

### TCN

In [ ]:
from src.models.nn.base_model import OutlierModelWrapper
from src.models.nn import TCN_train
import traceback
from lightning.pytorch.callbacks import EarlyStopping

import os
# ________________________________

from lightning.pytorch.callbacks import Callback


class HeartbeatCallback(Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        epoch = trainer.current_epoch + 1

        if epoch % 5 == 0:
            logging.info(f"HEARTBEAT: {pl_module.heartbeat}")
            

# ________________________________

# AVISO: POR ENQUANTO APENAS MODELOS DE RECONSTRUCAO
WINDOW_SIZE = 28
EPOCHS = 10
BATCH_SIZE = 512
PATIENCE = 5
THRESHOLD = 0.95
num_channels = [48, 64, 80, 80]
model_dict = {
    "TCN": TCN_train(WINDOW_SIZE, num_channels=num_channels, weight_decay=1e-6),
}
ACCEL = "cpu" if os.environ["AMBIENTE"] == "LOCAL" else "cuda"
DEV = os.environ["AMBIENTE"] == "LOCAL"
early_stop = EarlyStopping(monitor="val_f1", patience=PATIENCE)

wrapper_tcn = OutlierModelWrapper(
    dev=DEV,
    model_dict=model_dict,
    window_size=WINDOW_SIZE,
    lr=4e-4,
    threshold=THRESHOLD,
    batch_size=BATCH_SIZE,
    max_epochs=EPOCHS,
    accelerator=ACCEL,
    enable_progress_bar=not IS_BACKGROUND,
    enable_model_summary=not IS_BACKGROUND,
    trainer_callbacks=[HeartbeatCallback(), early_stop],
)


print("\n✓ Dataset carregado com sucesso!")
print(f"TAMANHO DO DATASET: {dataset.tam_dataset}")
# =====================================================
# TRAIN
# =====================================================
try:
    print("Iniciando pipeline do wrapper TCN...", flush=True)
    dict_met = wrapper_tcn.pipeline(train_values, train_labels, test_values, test_labels)
    print("Pipeline concluído com sucesso! Formatando resultados...", flush=True)
    # Achatar dicionário aninhado
    flat_records = []
    for nome, met in dict_met.items():
        flat_records.append(met)
    res = pd.DataFrame(flat_records)
    print(res)
except Exception as e:
    traceback.print_exc()
    print("ERRO AO TREINAR TCN!")
    raise e

### VAE

In [ ]:
from src.models.nn.base_model import OutlierModelWrapper
from src.models.nn import VAE
import traceback

# ________________________________

# AVISO: POR ENQUANTO APENAS MODELOS DE RECONSTRUCAO
model_dict = {
    "VAE": VAE(input_dim=WINDOW_SIZE),
}
wrapper_vae = OutlierModelWrapper(
    dev=DEV,
    model_dict=model_dict,
    window_size=WINDOW_SIZE,
    lr=1e-4,
    threshold=THRESHOLD,
    batch_size=BATCH_SIZE,
    max_epochs=EPOCHS,
    accelerator=ACCEL,
    enable_progress_bar=not IS_BACKGROUND,
    enable_model_summary=not IS_BACKGROUND,
    trainer_callbacks=[HeartbeatCallback(), early_stop],

)


print("\n✓ Dataset carregado com sucesso!")
print(f"TAMANHO DO DATASET: {dataset.tam_dataset}")
# =====================================================
# TRAIN
# =====================================================
try:
    dict_met = wrapper_vae.pipeline(train_values, train_labels, test_values, test_labels)

    # Achatar dicionário aninhado
    flat_records = []
    for nome, met in dict_met.items():
        flat_records.append(met)
    res = pd.DataFrame(flat_records)
    print(res)
except Exception as e:
    traceback.print_exc()
    print("ERRO AO TREINAR VAE!")
    raise e

## Métricas centralizadas

In [ ]:
from src.pipelines.metrics import CentralMetricsStore
import traceback
try:

    # Plota as m?tricas de valida??o e teste j? registradas pelos modelos
    figs_val = CentralMetricsStore.plot_all_metrics(split="validation")
    figs_test = CentralMetricsStore.plot_all_metrics(split="test")

    for fig in figs_val.values():
        fig.show()
        
    for name, fig in figs_test.items():
        if name in ["precision", "recall"]:
            continue 
        fig.show()


    for name, fig in CentralMetricsStore.plot_all_confusion_matrices().items():
        fig.show()
except:
    traceback.print_exc()
    print("ERRO PLOTAGEM DOS GRAFICOS")

## Visualizacao dos modelos

In [ ]:
# ============================================================
# VISUALIZAÇÃO: Scores de anomalia vs anotações reais
# ============================================================
from collections.abc import Sequence

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from darts import TimeSeries


SERIES_AMOUNT = 5  # <- ajuste aqui quantas séries visualizar

# -----------------------------------------------------------
# 1. Coletar scores de TODOS os modelos já treinados
# -----------------------------------------------------------
all_scores: dict[str, list[TimeSeries]] = {}

try:
        
    # Modelos baseline (objeto 'models' ainda vive no namespace)
    for model in models:
        name = model.__class__.__name__
        scores = model.test_scorer(test_values)
        all_scores.update(scores)

    # Modelos NN (wrapper ainda vive no namespace)
    nn_scores = wrapper_tcn.test_scorer(test_values)
    nn_scores["VAE"] = wrapper_vae.test_scorer(test_values)
    all_scores.update(nn_scores)

    model_names = list(all_scores.keys())
    print(f"Scores coletados de {len(model_names)} modelos: {model_names}")

    # -----------------------------------------------------------
    # 2. Selecionar séries — priorizar as que têm anomalias
    # -----------------------------------------------------------
    indices_com_anomalia = [i for i, lbl in enumerate(test_labels) if float(lbl.values(copy=False).sum()) > 0]
    indices_sem_anomalia = [i for i in range(len(test_values)) if i not in indices_com_anomalia]
    selected = (indices_com_anomalia + indices_sem_anomalia)[:SERIES_AMOUNT]

    print(f"Séries selecionadas (com anomalia primeiro): {selected}")

    # -----------------------------------------------------------
    # 3. Plotar uma figura por série
    # -----------------------------------------------------------
    for idx in selected:
        # Raw values da série
        raw_vals = test_values[idx].values(copy=False).flatten()
        x_vals = np.arange(len(raw_vals))

        # Ground truth
        labels = test_labels[idx].values(copy=False).flatten().astype(float)

        # Criar subplots: 1 linha para o sinal + 1 linha por modelo
        n_rows = 1 + len(model_names)
        fig = make_subplots(
            rows=n_rows,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.04,
            subplot_titles=(["Sinal original"] + [f"Score — {name}" for name in model_names]),
        )

        # --- Linha 1: sinal original + região anômala ---
        # Sombra onde há anomalia
        fig.add_trace(
            go.Scatter(
                x=x_vals,
                y=raw_vals,
                mode="lines",
                name="Sinal",
                line=dict(color="royalblue", width=1),
                hovertemplate="t=%{x}<br>valor=%{y:.4f}<extra></extra>",
            ),
            row=1,
            col=1,
        )
        # Overlay de anomalia (preenchimento vermelho semi-transparente)
        fig.add_trace(
            go.Scatter(
                x=x_vals,
                y=np.where(labels > 0, 1.0, None),
                mode="lines",
                name="Anomalia (GT)",
                line=dict(color="red", width=2),
                hovertemplate="t=%{x}<br>Anomalia<extra></extra>",
            ),
            row=1,
            col=1,
        )

        # --- Demais linhas: scores de cada modelo ---
        for row_idx, model_name in enumerate(model_names, start=2):
            score_vals = all_scores[model_name][idx].values(copy=False).flatten()

            # Alinhar comprimento (scores podem ser mais curtos devido ao windowing)
            if len(score_vals) < len(x_vals):
                pad = len(x_vals) - len(score_vals)
                score_vals = np.pad(score_vals, (pad, 0), constant_values=np.nan)

            # Score line
            fig.add_trace(
                go.Scatter(
                    x=x_vals,
                    y=score_vals,
                    mode="lines",
                    name=model_name,
                    line=dict(width=1.2),
                    hovertemplate="t=%{x}<br>score=%{y:.4f}<extra></extra>",
                    showlegend=False,
                ),
                row=row_idx,
                col=1,
            )

            # Região de anomalia (ground truth) sobreposta ao score
            fig.add_trace(
                go.Scatter(
                    x=x_vals,
                    y=np.where(labels > 0, 1.0, None),
                    mode="lines",
                    name="Anomalia (GT)",
                    line=dict(color="red", width=2),
                    showlegend=False,
                    hovertemplate="t=%{x}<br>Anomalia<extra></extra>",
                ),
                row=row_idx,
                col=1,
            )

        fig.update_layout(
            title=f"Série {idx} — Detecção de Anomalias (frente aos modelos)",
            height=200 + n_rows * 160,
            width=1300,
            template="plotly_white",
            hovermode="x unified",
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1,
            ),
        )

        # Eixos Y com rótulo
        fig.update_yaxes(title_text="Valor", row=1, col=1)
        for row_idx in range(2, n_rows + 1):
            fig.update_yaxes(title_text="Score", row=row_idx, col=1)
        fig.update_xaxes(title_text="Timestep", row=n_rows, col=1)

        fig.show()
except Exception:
    import traceback
    traceback.print_exc()

# Teste Otimização 2.0


In [ ]:
# ============================================================
# TESTE DA OTIMIZAÇÃO DE HIPERPARÂMETROS
# 10 trials de 1 época para validar o pipeline
# ============================================================

import optuna
import torch
torch.manual_seed(42)


from src.pipelines.optimization import HyperparamOptim
from src.models.nn.tcn import TCN_train
from src.models.nn.vae import VAE
from lightning.pytorch.callbacks import EarlyStopping


# --- Suggester de hiperparâmetros para TCN ---
def tcn_params(trial: optuna.Trial) -> dict:
    a = {
        "num_channels_0": trial.suggest_int("num_channels_0", 32, 64, step=16),
        "num_channels_1": trial.suggest_int("num_channels_1", 64, 128, step=8),
        "num_channels_2": trial.suggest_int("num_channels_2", 64, 128, step=8),
        "num_channels_3": trial.suggest_int("num_channels_3", 64, 128, step=8),
        "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        # "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "window_size": trial.suggest_int("window_size", 2, 50, step=1),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
    }

    return a


# --- Factory que constrói o model_dict a partir dos params ---
def tcn_factory(params: dict) -> dict:
    return {
        "TCN": TCN_train(
            window_size=params["window_size"],
            dropout=0.3,
            num_channels=[
                params["num_channels_0"],
                params["num_channels_1"],
                params["num_channels_2"],
                params["num_channels_3"],
            ],
            lr=params["lr"],
            weight_decay=params["weight_decay"],
        )
    }


from lightning.pytorch.callbacks import Callback


class HeartbeatCallback(Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        epoch = trainer.current_epoch + 1

        if epoch % 5 == 0:
            print(f"HEARTBEAT: {pl_module.heartbeat}", flush=True)


logging.info("OTIMIZANDO TCN")

ACCEL = "cpu" if os.environ["AMBIENTE"] == "LOCAL" else "cuda"
THRESHOLD = 0.90
MAX_EPOCHS = 3
PATIENCE = 5
N_TRIALS = 60
DIRECTION = "maximize"
NAME = "test_tcn_opt"
BATCH_SZIE = 512
early_stop = EarlyStopping(monitor="val_f1", patience=PATIENCE)

# --- Instancia e roda ---
optim = HyperparamOptim(
    model_factory=tcn_factory,
    param_suggester=tcn_params,
    train_data=(train_values, train_labels),
    test_data=(test_values, test_labels),
    fixed_params={"accelerator": ACCEL, "threshold": THRESHOLD, "dev": True},
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    n_trials=N_TRIALS,
    direction=DIRECTION,
    study_name=NAME,
    callbacks=[HeartbeatCallback(), early_stop],
)
# timeout 2h

study = optim.optimize(timeout=60 * 60 * 2)


# --- Suggester de hiperparâmetros para VAE ---
def vae_params(trial: optuna.Trial) -> dict:
    a = {
        "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        # "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "hidden_dim": trial.suggest_int("hidden_dim", 500, 1000, step=10),
        "latent_dim": trial.suggest_int("latent_dim", 500, 1000, step=10),
        "window_size": trial.suggest_int("window_size", 2, 50, step=1),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
    }

    return a


# --- Factory que constrói o model_dict a partir dos params ---
def vae_factory(params: dict) -> dict:
    return {
        "VAE": VAE(
            input_dim=params["window_size"],
            hidden_dim=params["hidden_dim"],
            latent_dim=params["latent_dim"],
            lr=params["lr"],
            weight_decay=params["weight_decay"],
        )
    }


early_stop = EarlyStopping(monitor="val_f1", patience=PATIENCE)

logging.info("OTIMIZANDO VAE")
NAME = "test_vae_opt"
# --- Instancia e roda ---
optim = HyperparamOptim(
    model_factory=vae_factory,
    param_suggester=vae_params,
    train_data=(train_values, train_labels),
    test_data=(test_values, test_labels),
    fixed_params={"accelerator": ACCEL, "threshold": THRESHOLD},
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    n_trials=N_TRIALS,
    batch_size=BATCH_SIZE,
    direction=DIRECTION,
    study_name=NAME,
    callbacks=[HeartbeatCallback(), early_stop],
)
# timeout 2h
study = optim.optimize(timeout=60 * 60 * 2)